# Layer 3 — Urban Growth Forecasting (2026–2036)

CA-Markov + Gradient Boosted Trees hybrid model:
1. **Markov Chain** — transition probability matrices from historical LULC
2. **GBT suitability** — gradient boosting on spectral + proximity features
3. **Cellular Automata** — iterative prediction with feature updates
4. **Backtest** — validate by predicting 2021 from 2016, compare to actual
5. **Forecast** — predicted LULC maps for 2026, 2031, 2036

All processing in GEE. Exports to Drive → `guwahati_urban_prediction/`

## 0. Setup

In [ ]:
import ee

ee.Authenticate()
ee.Initialize(project='vivid-brand-182819')
print('Earth Engine initialized:', ee.String('connected').getInfo())

BBOX = {'west': 91.55, 'south': 26.10, 'east': 91.88, 'north': 26.25}
aoi = ee.Geometry.Rectangle([BBOX['west'], BBOX['south'], BBOX['east'], BBOX['north']])

LULC_CLASSES = {0: 'Urban', 1: 'Vegetation', 2: 'Water', 3: 'Barren'}
NUM_CLASSES = 4
DRIVE_FOLDER = 'guwahati_urban_prediction'
SCALE = 30

print(f'AOI area: {aoi.area().divide(1e6).getInfo():.1f} km²')

## 1. Rebuild LULC maps & features from Layer 2

In [ ]:
# ── Composite builders (same as Layer 2) ──

def cloud_mask_s2(image):
    scl = image.select('SCL')
    clear = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11))
    return image.updateMask(clear)

def add_indices(image, nir='B8', red='B4', swir='B11', green='B3'):
    ndvi = image.normalizedDifference([nir, red]).rename('NDVI')
    ndbi = image.normalizedDifference([swir, nir]).rename('NDBI')
    ndwi = image.normalizedDifference([green, nir]).rename('NDWI')
    return image.addBands([ndvi, ndbi, ndwi])

def cloud_mask_landsat(image):
    qa = image.select('QA_PIXEL')
    return image.updateMask(qa.bitwiseAnd(1 << 3).eq(0).And(qa.bitwiseAnd(1 << 4).eq(0)))

def scale_l8(image):
    optical = image.select(['SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7']).multiply(0.0000275).add(-0.2)
    return image.addBands(optical, overwrite=True)

def scale_l5(image):
    optical = image.select(['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5','SR_B7']).multiply(0.0000275).add(-0.2)
    return image.addBands(optical, overwrite=True)

def build_landsat_composite(start, end, sensor='L8'):
    if sensor == 'L8':
        col_id, sfn = 'LANDSAT/LC08/C02/T1_L2', scale_l8
        bands = ['SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7']
        nir, red, swir, green = 'SR_B5', 'SR_B4', 'SR_B6', 'SR_B3'
    else:
        col_id, sfn = 'LANDSAT/LT05/C02/T1_L2', scale_l5
        bands = ['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5','SR_B7']
        nir, red, swir, green = 'SR_B4', 'SR_B3', 'SR_B5', 'SR_B2'
    _nir, _red, _swir, _green = nir, red, swir, green
    def _add(img, _n=_nir, _r=_red, _s=_swir, _g=_green):
        return add_indices(img, _n, _r, _s, _g)
    col = (ee.ImageCollection(col_id).filterBounds(aoi).filterDate(start, end)
           .filter(ee.Filter.lt('CLOUD_COVER', 30)).map(cloud_mask_landsat).map(sfn).map(_add))
    return col.select(bands + ['NDVI','NDBI','NDWI']).median().clip(aoi).toFloat()

# ── LULC classifier (same as Layer 2) ──

def generate_training(composite):
    ndvi, ndbi, ndwi = composite.select('NDVI'), composite.select('NDBI'), composite.select('NDWI')
    urban = ndbi.gt(0.0).And(ndvi.lt(0.25))
    veg = ndvi.gt(0.35).And(ndbi.lt(0.0))
    water = ndwi.gt(0.1).And(ndvi.lt(0.15))
    barren = ndvi.lt(0.15).And(ndbi.lt(0.05)).And(ndwi.lt(0.0))
    labels = ee.Image(0).where(urban,0).where(veg,1).where(water,2).where(barren,3).rename('label').toUint8()
    labels = labels.updateMask(urban.Or(veg).Or(water).Or(barren))
    return composite.addBands(labels).stratifiedSample(
        numPoints=500, classBand='label', region=aoi, scale=SCALE, seed=42, geometries=True)

CLASSIFY_BANDS = ['NDVI', 'NDBI', 'NDWI']

# ── Proximity feature builders ──

def proximity_stack(lulc_img):
    water = lulc_img.eq(2)
    dist_water = water.fastDistanceTransform(256).sqrt().multiply(SCALE).rename('dist_water')
    urban = lulc_img.eq(0)
    edge = urban.And(urban.focal_min(1).Not())
    dist_edge = edge.fastDistanceTransform(256).sqrt().multiply(SCALE).rename('dist_urban_edge')
    viirs = ee.ImageCollection('NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG').filterBounds(aoi).filterDate('2021-01-01','2021-12-31').select('avg_rad').median().clip(aoi)
    lit = viirs.gt(5).unmask(0)
    dist_road = lit.fastDistanceTransform(256).sqrt().multiply(SCALE).rename('dist_road')
    srtm = ee.Image('USGS/SRTMGL1_003').clip(aoi)
    slope = ee.Terrain.slope(srtm).rename('slope')
    elev = srtm.select('elevation')
    return dist_water.addBands(dist_edge).addBands(dist_road).addBands(slope).addBands(elev).toFloat()

# ── Build composites and classify all epochs ──

EPOCHS = {
    '2000': ('2000-11-01','2001-04-30','L5'),
    '2005': ('2005-11-01','2006-04-30','L5'),
    '2010': ('2010-11-01','2011-04-30','L5'),
    '2016': ('2016-11-01','2017-04-30','L8'),
    '2021': ('2021-11-01','2022-04-30','L8'),
}

composites = {}
for lbl, (s, e, sen) in EPOCHS.items():
    composites[lbl] = build_landsat_composite(s, e, sen)
    print(f'[Composite] {lbl} ready')

# Train CART on 2021 reference
samples_2021 = generate_training(composites['2021'])
cart = ee.Classifier.smileCart(maxNodes=50).train(samples_2021, 'label', CLASSIFY_BANDS)

lulc_maps = {}
for lbl, comp in composites.items():
    lulc_maps[lbl] = comp.select(CLASSIFY_BANDS).classify(cart).toUint8().rename('lulc')
    print(f'[LULC] {lbl} classified')

print(f'\nAll {len(lulc_maps)} LULC maps and composites ready')

## 2. Markov transition matrices

In [ ]:
# Compute Markov transition matrices from consecutive LULC epoch pairs
# Matrix[i][j] = probability of class i → class j in one epoch step (~5 years)

import numpy as np

epoch_pairs = [('2000','2005'), ('2005','2010'), ('2010','2016'), ('2016','2021')]

def compute_transition_matrix(lulc_from, lulc_to, n_classes=4):
    """Compute transition counts via GEE reduceRegion, return probability matrix."""
    # Encode transitions as from*10 + to
    transition = lulc_from.multiply(10).add(lulc_to).rename('transition')
    pixel_area = ee.Image.pixelArea()

    matrix = np.zeros((n_classes, n_classes))
    for i in range(n_classes):
        for j in range(n_classes):
            code = i * 10 + j
            area = transition.eq(code).multiply(pixel_area).reduceRegion(
                reducer=ee.Reducer.sum(), geometry=aoi, scale=SCALE, maxPixels=1e9
            ).get('transition')
            matrix[i][j] = ee.Number(area).getInfo()

    # Normalize rows to probabilities
    row_sums = matrix.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1  # avoid div-by-zero
    return matrix / row_sums

transition_matrices = {}
for e1, e2 in epoch_pairs:
    print(f'Computing transition matrix {e1} → {e2}...')
    tm = compute_transition_matrix(lulc_maps[e1], lulc_maps[e2])
    transition_matrices[f'{e1}_{e2}'] = tm
    print(f'  {e1}→{e2}:')
    for i, cls in LULC_CLASSES.items():
        row = ' '.join([f'{tm[i][j]:.3f}' for j in range(NUM_CLASSES)])
        print(f'    {cls:12s} → [{row}]')
    print()

# Average transition matrix (used for forward projection)
avg_tm = np.mean(list(transition_matrices.values()), axis=0)
print('=== Average Transition Matrix (per 5-year step) ===')
header = '             ' + '  '.join([f'{c:>10s}' for c in LULC_CLASSES.values()])
print(header)
for i, cls in LULC_CLASSES.items():
    row = '  '.join([f'{avg_tm[i][j]:10.4f}' for j in range(NUM_CLASSES)])
    print(f'{cls:12s}  {row}')

# Key metric: probability of non-urban → urban
print(f'\nVeg→Urban rate:    {avg_tm[1][0]:.4f} per 5yr')
print(f'Barren→Urban rate: {avg_tm[3][0]:.4f} per 5yr')
print(f'Urban persistence: {avg_tm[0][0]:.4f}')

## 3. Train GBT on historical urban transitions

In [ ]:
# Train Gradient Boosted Trees on urban-change transitions
# Label: did this pixel become urban in the next epoch? (binary)
# Features: spectral indices + proximity + terrain from the "before" epoch

# Build training data from 2010→2016 and 2016→2021 transitions
FEATURE_BANDS = ['NDVI', 'NDBI', 'NDWI', 'dist_water', 'dist_urban_edge', 'dist_road', 'slope', 'elevation']

def build_change_samples(lulc_before, lulc_after, composite_before, lulc_before_img, n_pos=1000, n_neg=2000):
    """Sample pixels that did/didn't convert to urban."""
    prox = proximity_stack(lulc_before_img)
    features = composite_before.select(['NDVI','NDBI','NDWI']).addBands(prox)

    # Positive: non-urban → urban
    became_urban = lulc_before.neq(0).And(lulc_after.eq(0)).rename('became_urban').toUint8()
    # Negative: stayed non-urban
    stayed_nonurban = lulc_before.neq(0).And(lulc_after.neq(0)).rename('became_urban')
    stayed_nonurban = stayed_nonurban.multiply(0).toUint8()  # label = 0

    pos_input = features.addBands(became_urban.updateMask(became_urban))
    neg_input = features.addBands(stayed_nonurban.updateMask(lulc_before.neq(0).And(lulc_after.neq(0))))

    pos_samples = pos_input.sample(region=aoi, scale=SCALE, numPixels=n_pos, seed=42, geometries=True)
    neg_samples = neg_input.sample(region=aoi, scale=SCALE, numPixels=n_neg, seed=43, geometries=True)

    return pos_samples.merge(neg_samples)

# Collect training samples from two epoch transitions
print('Sampling 2010→2016 transitions...')
samples_1 = build_change_samples(lulc_maps['2010'], lulc_maps['2016'],
                                  composites['2010'], lulc_maps['2010'])
print(f'  Samples: {samples_1.size().getInfo()}')

print('Sampling 2016→2021 transitions...')
samples_2 = build_change_samples(lulc_maps['2016'], lulc_maps['2021'],
                                  composites['2016'], lulc_maps['2016'])
print(f'  Samples: {samples_2.size().getInfo()}')

all_samples = samples_1.merge(samples_2)
total = all_samples.size().getInfo()
print(f'\nTotal training samples: {total}')

# Train GBT classifier
gbt = ee.Classifier.smileGradientTreeBoost(
    numberOfTrees=100,
    shrinkage=0.1,
    maxNodes=20,
    seed=42
).train(
    features=all_samples,
    classProperty='became_urban',
    inputProperties=FEATURE_BANDS,
)

# Switch to probability mode for suitability scoring
gbt_prob = gbt.setOutputMode('PROBABILITY')

print('GBT classifier trained (100 trees, lr=0.1)')
print(f'Features: {FEATURE_BANDS}')

## 4. Backtest — predict 2021 from 2016, compare to actual

In [ ]:
# Backtest: use 2016 features to predict 2021, compare to actual 2021 LULC
# This validates both the GBT model and the CA-Markov allocation logic

prox_2016 = proximity_stack(lulc_maps['2016'])
feat_2016 = composites['2016'].select(['NDVI','NDBI','NDWI']).addBands(prox_2016)

# GBT probability of becoming urban
suit_2016 = feat_2016.classify(gbt_prob).rename('urban_prob')

# Only non-urban pixels can convert
non_urban_2016 = lulc_maps['2016'].neq(0)
suit_2016 = suit_2016.updateMask(non_urban_2016)

# How many pixels should convert? Use Markov rate from 2016→2021
tm_recent = transition_matrices['2016_2021']

# Count current non-urban pixels by class
pixel_area = ee.Image.pixelArea()
total_area = {}
for cls_id in range(NUM_CLASSES):
    a = lulc_maps['2016'].eq(cls_id).multiply(pixel_area).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=aoi, scale=SCALE, maxPixels=1e9
    ).get('lulc')
    total_area[cls_id] = ee.Number(a).getInfo()

# Expected new urban area from Markov rates
expected_new_urban_m2 = sum(total_area[c] * tm_recent[c][0] for c in range(1, NUM_CLASSES))
expected_new_urban_pixels = int(expected_new_urban_m2 / (SCALE * SCALE))
print(f'Expected new urban pixels (Markov): {expected_new_urban_pixels:,}')
print(f'Expected new urban area: {expected_new_urban_m2/1e6:.2f} km²')

# Allocate: top-N suitability pixels become urban
# Use percentile threshold to select top pixels
threshold = suit_2016.reduceRegion(
    reducer=ee.Reducer.percentile([95]),
    geometry=aoi, scale=SCALE, maxPixels=1e9
).get('urban_prob')
threshold_val = ee.Number(threshold).getInfo()
print(f'Suitability threshold (95th pctl): {threshold_val:.4f}')

# Predicted 2021: existing urban + newly converted
predicted_urban = suit_2016.gt(threshold_val).unmask(0)
predicted_2021 = lulc_maps['2016'].where(
    non_urban_2016.And(predicted_urban), 0
)

# Compare to actual 2021
actual_2021 = lulc_maps['2021']
correct = predicted_2021.eq(actual_2021)
accuracy = correct.reduceRegion(
    reducer=ee.Reducer.mean(), geometry=aoi, scale=SCALE, maxPixels=1e9
).get('lulc')
print(f'\nBacktest overall accuracy: {ee.Number(accuracy).getInfo():.4f}')

# Urban-specific accuracy
actual_urban = actual_2021.eq(0)
pred_urban = predicted_2021.eq(0)
tp = actual_urban.And(pred_urban).reduceRegion(ee.Reducer.sum(), aoi, SCALE, maxPixels=1e9).get('lulc')
fp = pred_urban.And(actual_urban.Not()).reduceRegion(ee.Reducer.sum(), aoi, SCALE, maxPixels=1e9).get('lulc')
fn = actual_urban.And(pred_urban.Not()).reduceRegion(ee.Reducer.sum(), aoi, SCALE, maxPixels=1e9).get('lulc')
tp_v = ee.Number(tp).getInfo()
fp_v = ee.Number(fp).getInfo()
fn_v = ee.Number(fn).getInfo()
precision = tp_v / (tp_v + fp_v) if (tp_v + fp_v) > 0 else 0
recall = tp_v / (tp_v + fn_v) if (tp_v + fn_v) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
print(f'Urban precision: {precision:.4f}')
print(f'Urban recall:    {recall:.4f}')
print(f'Urban F1:        {f1:.4f}')

## 5. CA-Markov iterative forecast (2026, 2031, 2036)

In [ ]:
# CA-Markov iterative forecast: 2021 → 2026 → 2031 → 2036
# Each step: compute suitability → allocate new urban pixels (Markov-constrained) → update LULC

def forecast_one_step(current_lulc, composite, markov_tm, gbt_classifier):
    """Predict next epoch's LULC using CA-Markov + GBT hybrid."""
    prox = proximity_stack(current_lulc)
    features = composite.select(['NDVI','NDBI','NDWI']).addBands(prox)

    # GBT suitability for urban conversion
    suitability = features.classify(gbt_classifier).rename('urban_prob')
    non_urban = current_lulc.neq(0)
    suitability = suitability.updateMask(non_urban)

    # Count current class areas for Markov allocation
    pixel_area_img = ee.Image.pixelArea()
    class_areas = {}
    for c in range(NUM_CLASSES):
        a = current_lulc.eq(c).multiply(pixel_area_img).reduceRegion(
            reducer=ee.Reducer.sum(), geometry=aoi, scale=SCALE, maxPixels=1e9
        ).get('lulc')
        class_areas[c] = ee.Number(a).getInfo()

    # Expected new urban area from non-urban classes
    new_urban_m2 = sum(class_areas[c] * markov_tm[c][0] for c in range(1, NUM_CLASSES))
    new_urban_frac = new_urban_m2 / sum(class_areas[c] for c in range(1, NUM_CLASSES)) if sum(class_areas[c] for c in range(1, NUM_CLASSES)) > 0 else 0

    # Compute threshold: what percentile of suitability captures the right amount of area
    target_pctl = max(0, min(99, (1 - new_urban_frac) * 100))
    threshold = suitability.reduceRegion(
        reducer=ee.Reducer.percentile([target_pctl]),
        geometry=aoi, scale=SCALE, maxPixels=1e9
    ).get('urban_prob')
    threshold_val = ee.Number(threshold).getInfo()

    # Allocate: pixels above threshold become urban
    new_urban_mask = suitability.gt(threshold_val).unmask(0)

    # Water class is protected (Brahmaputra, Deepor Beel don't urbanize)
    water_protected = current_lulc.eq(2)
    new_urban_mask = new_urban_mask.And(water_protected.Not())

    # Update LULC
    next_lulc = current_lulc.where(non_urban.And(new_urban_mask), 0)

    return next_lulc, new_urban_m2

# Use the 2021 composite as the spectral baseline for future predictions
# (spectral indices won't change drastically over 5-year steps for this PoC)
base_composite = composites['2021']

predicted_lulc = {'2021': lulc_maps['2021']}
forecast_years = ['2026', '2031', '2036']

current = lulc_maps['2021']
for year in forecast_years:
    print(f'\nForecasting {year}...')
    next_lulc, new_area = forecast_one_step(current, base_composite, avg_tm, gbt_prob)
    predicted_lulc[year] = next_lulc
    print(f'  New urban area added: {new_area/1e6:.2f} km²')

    # Report total urban area
    urban_total = next_lulc.eq(0).multiply(ee.Image.pixelArea()).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=aoi, scale=SCALE, maxPixels=1e9
    ).get('lulc')
    print(f'  Total urban area {year}: {ee.Number(urban_total).divide(1e6).getInfo():.2f} km²')

    current = next_lulc

print('\nForecasting complete!')

## 6. Export predicted LULC maps

In [ ]:
# Export predicted LULC maps + suitability surface

forecast_tasks = []

for year in forecast_years:
    task = ee.batch.Export.image.toDrive(
        image=predicted_lulc[year],
        description=f'predicted_lulc_{year}',
        folder=DRIVE_FOLDER,
        fileNamePrefix=f'predicted_lulc_{year}',
        region=aoi,
        scale=SCALE,
        crs='EPSG:4326',
        maxPixels=1e9,
        fileFormat='GeoTIFF',
    )
    task.start()
    forecast_tasks.append(task)
    print(f'[Export] predicted_lulc_{year}')

# Export the 2021 suitability surface (useful for visualization in Layer 5)
prox_2021 = proximity_stack(lulc_maps['2021'])
feat_2021 = composites['2021'].select(['NDVI','NDBI','NDWI']).addBands(prox_2021)
suit_2021 = feat_2021.classify(gbt_prob).rename('urban_prob').toFloat()

task_suit = ee.batch.Export.image.toDrive(
    image=suit_2021,
    description='suitability_2021',
    folder=DRIVE_FOLDER,
    fileNamePrefix='suitability_2021',
    region=aoi,
    scale=SCALE,
    crs='EPSG:4326',
    maxPixels=1e9,
    fileFormat='GeoTIFF',
    formatOptions={'cloudOptimized': True},
)
task_suit.start()
forecast_tasks.append(task_suit)
print('[Export] suitability_2021')

# Export backtest prediction for validation comparison
task_bt = ee.batch.Export.image.toDrive(
    image=predicted_2021,
    description='backtest_predicted_2021',
    folder=DRIVE_FOLDER,
    fileNamePrefix='backtest_predicted_2021',
    region=aoi,
    scale=SCALE,
    crs='EPSG:4326',
    maxPixels=1e9,
    fileFormat='GeoTIFF',
)
task_bt.start()
forecast_tasks.append(task_bt)
print('[Export] backtest_predicted_2021')

print(f'\n{len(forecast_tasks)} forecast export tasks submitted')

## 7. Urban growth summary statistics

In [ ]:
# Urban growth summary — historical + predicted

print('=== Guwahati Urban Growth Timeline ===\n')
print(f'{"Year":<8} {"Urban km²":>12} {"Status":>12}')
print('-' * 36)

all_years = list(lulc_maps.keys()) + forecast_years
all_lulc = {**lulc_maps, **{y: predicted_lulc[y] for y in forecast_years}}

prev_area = None
for year in sorted(all_years, key=int):
    lulc = all_lulc[year]
    area = lulc.eq(0).multiply(ee.Image.pixelArea()).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=aoi, scale=SCALE, maxPixels=1e9
    ).get('lulc')
    area_km2 = ee.Number(area).divide(1e6).getInfo()

    status = 'historical' if int(year) <= 2021 else 'predicted'
    growth = f' (+{area_km2 - prev_area:.1f})' if prev_area else ''
    print(f'{year:<8} {area_km2:>10.2f}{growth:>14} {status}')
    prev_area = area_km2

print(f'\n{"="*36}')
first = all_years[0]
last = forecast_years[-1]
print(f'Total growth {first}→{last}: predicted above')

## 8. Check export task status

In [ ]:
# Check status of all Layer 3 export tasks

all_l3_tasks = forecast_tasks
print(f'Total Layer 3 export tasks: {len(all_l3_tasks)}')
print('Monitor at: https://code.earthengine.google.com/tasks\n')

completed = 0
for t in all_l3_tasks:
    status = t.status()
    state = status['state']
    desc = status['description']
    err = status.get('error_message', '')
    line = f'  {state:12s} {desc}'
    if err:
        line += f'  ERROR: {err}'
    print(line)
    if state == 'COMPLETED':
        completed += 1

print(f'\n{completed}/{len(all_l3_tasks)} completed')
if completed < len(all_l3_tasks):
    print('Re-run this cell to refresh status.')